<a href="https://colab.research.google.com/github/alinesaurer826-bot/trabalhocolab/blob/main/Workshop_Chatbot_RAG_HuggingFace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop — Chatbot com RAG usando Hugging Face

Em ~15 minutos vamos construir um **chatbot que responde com base em documentos**, usando apenas modelos abertos do Hugging Face rodando na GPU do Colab. Não precisa de chave de API e não tem custo.

## O problema

Um LLM só sabe o que aprendeu no treinamento. Se perguntarmos sobre as regras internas de uma empresa, ele **não sabe**, e muitas vezes **inventa** uma resposta convincente (isso se chama *alucinação*).

## A solução: RAG (*Retrieval-Augmented Generation*)

RAG significa "geração aumentada por recuperação". Em vez de treinar o modelo de novo, nós **buscamos os trechos relevantes** dos nossos documentos e os **entregamos junto com a pergunta**:

```
                  ┌──────────────────────────┐
  Pergunta ──────►│ 1. BUSCA (Retrieval)     │  encontra os trechos mais
                  │    na base de documentos │  parecidos com a pergunta
                  └────────────┬─────────────┘
                               │ trechos relevantes
                  ┌────────────▼─────────────┐
                  │ 2. PROMPT AUMENTADO      │  "Use este contexto para
                  │    contexto + pergunta   │   responder a pergunta..."
                  └────────────┬─────────────┘
                  ┌────────────▼─────────────┐
                  │ 3. GERAÇÃO (LLM)         │──────► Resposta fundamentada
                  └──────────────────────────┘
```

## Roteiro

1. Instalação
2. Base de conhecimento → dividir em pedaços (*chunks*)
3. Embeddings e banco vetorial (a "busca inteligente")
4. Carregar o LLM
5. Comparar: **sem RAG × com RAG**
6. Chatbot com RAG

> ⚙️ Antes de começar: *Ambiente de execução → Alterar o tipo de ambiente de execução → **GPU T4***. Depois execute as células em ordem, com `Shift + Enter`.

## 1. Instalação

| Pacote | Função |
|--------|--------|
| `langchain-huggingface` | Conecta o LangChain aos modelos do Hugging Face (LLM e embeddings) |
| `langchain-text-splitters` | Divide textos longos em pedaços |
| `sentence-transformers` | Gera os *embeddings* (vetores) dos textos |

> O aviso vermelho do pip sobre `google-colab` pode ser ignorado.

In [ ]:
!pip install -q -U langchain-huggingface langchain-text-splitters sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 24.2 MB/s eta 0:00:00


## 2. Base de conhecimento

Nossa "fonte da verdade" será o FAQ interno de uma empresa **fictícia**, a *TechNova*. Como ela não existe, o modelo **não tem como saber** essas informações — perfeito para mostrar o efeito do RAG.

Em um projeto real, esse texto viria de PDFs, páginas web, planilhas ou um banco de dados.

In [ ]:
base_conhecimento = '''
Horário de trabalho
O expediente na TechNova é das 9h às 18h, com 1 hora de almoço. Existe flexibilidade de entrada entre 8h e 10h, desde que a jornada de 8 horas seja cumprida.

Home office
Os colaboradores podem trabalhar remotamente às segundas e sextas-feiras. Terças, quartas e quintas são dias presenciais obrigatórios no escritório da Avenida Paulista.

Férias
As férias devem ser solicitadas pelo portal RH Fácil com no mínimo 45 dias de antecedência. É possível dividir as férias em até 3 períodos, sendo que um deles deve ter pelo menos 14 dias.

Vale-refeição
O vale-refeição é de R$ 50,00 por dia útil, creditado no cartão Alimenta+ todo dia 25 do mês anterior.

Suporte de TI
Problemas com computador, e-mail ou acesso devem ser abertos no sistema HelpTech ou pelo ramal 4040. O atendimento funciona das 8h às 20h. Senhas bloqueadas são resetadas em até 30 minutos.

Treinamentos
Cada colaborador tem direito a um orçamento anual de R$ 3.000,00 para cursos e certificações. O pedido deve ser aprovado pelo gestor direto antes da inscrição.

Equipamentos
Todo colaborador recebe um notebook e um headset no primeiro dia. Monitores adicionais podem ser solicitados para o home office pelo HelpTech, com entrega em até 10 dias úteis.
'''
print(f"Base com {len(base_conhecimento)} caracteres.")

Base com 1245 caracteres.


### Dividindo em pedaços (*chunking*)

Não enviamos o documento inteiro para o modelo, por três motivos: o modelo tem um limite de texto (*janela de contexto*), textos longos deixam a resposta mais lenta e trechos menores tornam a busca mais precisa.

O `RecursiveCharacterTextSplitter` corta o texto tentando respeitar parágrafos e frases:

- `chunk_size=300` → cada pedaço tem no máximo ~300 caracteres.
- `chunk_overlap=30` → pedaços vizinhos compartilham um trecho, para não cortar uma ideia ao meio.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

divisor = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = divisor.create_documents([base_conhecimento])

print(f"{len(chunks)} chunks criados. Exemplo:\n")
print(chunks[0].page_content)

7 chunks criados. Exemplo:

Horário de trabalho
O expediente na TechNova é das 9h às 18h, com 1 hora de almoço. Existe flexibilidade de entrada entre 8h e 10h, desde que a jornada de 8 horas seja cumprida.


## 3. Embeddings e banco vetorial

Como o computador sabe que *"posso trabalhar de casa?"* tem a ver com o trecho *"Home office"*, se as palavras são diferentes?

Com **embeddings**: um modelo transforma cada texto em um **vetor de números** que representa o seu *significado*. Textos com significados parecidos geram vetores próximos.

```
"trabalhar de casa"   →  [0.12, -0.48, 0.91, ...]  ┐
"home office"         →  [0.10, -0.45, 0.88, ...]  ┘ próximos!
"vale-refeição"       →  [-0.70, 0.33, 0.05, ...]    distante
```

Usamos o modelo `paraphrase-multilingual-MiniLM-L12-v2`, pequeno e que entende português. Os vetores ficam guardados em um **banco vetorial** — aqui, um simples `InMemoryVectorStore` (na memória). Em produção usaríamos FAISS, Chroma, pgvector etc.

In [ ]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
print("Usando:", dispositivo)

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": dispositivo},
)

banco_vetorial = InMemoryVectorStore.from_documents(chunks, embeddings)
print(f"✅ {len(chunks)} chunks indexados no banco vetorial.")

Usando: cuda


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ 7 chunks indexados no banco vetorial.


### Testando só a busca

Antes de envolver o LLM, veja a busca funcionando sozinha. Repare que a pergunta **não usa as mesmas palavras** do texto, e mesmo assim o trecho certo aparece no topo. O *score* indica a similaridade (quanto maior, mais parecido).

In [ ]:
pergunta_teste = "Posso trabalhar de casa na quarta-feira?"

for doc, score in banco_vetorial.similarity_search_with_score(pergunta_teste, k=2):
    print(f"[score {score:.2f}] {doc.page_content}\n")

[score 0.65] Home office
Os colaboradores podem trabalhar remotamente às segundas e sextas-feiras. Terças, quartas e quintas são dias presenciais obrigatórios no escritório da Avenida Paulista.

[score 0.43] Horário de trabalho
O expediente na TechNova é das 9h às 18h, com 1 hora de almoço. Existe flexibilidade de entrada entre 8h e 10h, desde que a jornada de 8 horas seja cumprida.



## 4. Carregando o LLM

Usamos o `Qwen2.5-1.5B-Instruct`: um modelo aberto, pequeno o bastante para a GPU gratuita do Colab, versão *Instruct* (treinada para conversar) e bom em português. A primeira execução baixa ~3 GB e leva alguns minutos.

- `HuggingFacePipeline` carrega o modelo na GPU.
- `ChatHuggingFace` permite conversar com ele usando mensagens (sistema / usuário / assistente).
- `do_sample=False` faz o modelo escolher sempre a palavra mais provável. Para respostas **factuais**, queremos consistência, não criatividade.

In [ ]:
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace

llm = HuggingFacePipeline.from_model_id(
    model_id="Qwen/Qwen2.5-1.5B-Instruct",
    task="text-generation",
    device=0 if torch.cuda.is_available() else -1,
    pipeline_kwargs={"max_new_tokens": 200, "do_sample": False, "return_full_text": False},
)
chat = ChatHuggingFace(llm=llm)
print("✅ LLM carregado.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ LLM carregado.


## 5. O teste decisivo: sem RAG × com RAG

### 5.1 Sem RAG

Perguntamos direto ao modelo. Ele não conhece a TechNova, então vai responder de forma genérica ou **inventar**.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

pergunta = "Quanto é o vale-refeição na TechNova e quando ele é creditado?"

resposta = chat.invoke([HumanMessage(content=pergunta)])
print(resposta.text)

Desculpe, mas não tenho informações específicas sobre o vale-refeição da TechNova ou como ele é creditado. Isso poderia variar dependendo do plano de benefícios individualmente definido pela empresa.

Para obter detalhes precisos sobre seu vale-refeição na TechNova, sugiro que você:

1. Ligue para a assistência operacional da empresa.
2. Consulte diretamente o site oficial da TechNova se houver um disponível.
3. Procure em documentos pessoais ou contratos de trabalho que possam ter essas informações.

Se você for trabalhar com uma equipe específica ou departamento específico dentro da TechNova, também pode ser útil entrar em contato com eles diretamente para obter as informações mais precisas.


### 5.2 Com RAG

Agora a função `responder_com_rag` faz as três etapas do diagrama:

1. **Busca** os 3 chunks mais parecidos com a pergunta (`retriever`).
2. **Monta o prompt aumentado**: uma instrução de sistema com o contexto encontrado e a regra *"responda apenas com base no contexto"*. Essa regra é o que reduz a alucinação.
3. **Gera** a resposta com o LLM.

Também devolvemos as **fontes** (os trechos usados), algo essencial em aplicações reais para o usuário poder conferir a resposta.

In [ ]:
retriever = banco_vetorial.as_retriever(search_kwargs={"k": 3})

INSTRUCAO_RAG = (
    "Você é o assistente de RH da empresa TechNova. "
    "Responda em português, de forma curta e direta, usando APENAS as informações do contexto abaixo. "
    "Se a resposta não estiver no contexto, diga: 'Não encontrei essa informação nos documentos.'\n\n"
    "Contexto:\n{contexto}"
)

def responder_com_rag(pergunta, historico=None):
    # 1. BUSCA
    docs = retriever.invoke(pergunta)
    contexto = "\n\n".join(doc.page_content for doc in docs)

    # 2. PROMPT AUMENTADO (sistema com contexto + histórico recente + pergunta)
    mensagens = [SystemMessage(content=INSTRUCAO_RAG.format(contexto=contexto))]
    mensagens += (historico or [])
    mensagens.append(HumanMessage(content=pergunta))

    # 3. GERAÇÃO
    resposta = chat.invoke(mensagens)
    return resposta.text, docs

resposta, fontes = responder_com_rag(pergunta)
print("Resposta:", resposta)
print("\nFontes usadas:")
for doc in fontes:
    print(" •", doc.page_content[:80], "...")

Resposta: O vale-refeição na TechNova é de R$ 50,00 por dia útil, creditado no cartão Alimenta+ todo dia 25 do mês anterior.

Fontes usadas:
 • Vale-refeição
O vale-refeição é de R$ 50,00 por dia útil, creditado no cartão Al ...
 • Horário de trabalho
O expediente na TechNova é das 9h às 18h, com 1 hora de almo ...
 • Treinamentos
Cada colaborador tem direito a um orçamento anual de R$ 3.000,00 pa ...


Agora a resposta traz o valor e a data corretos, tirados do documento.

E se perguntarmos algo que **não está** na base? O modelo deve admitir que não sabe, em vez de inventar:

In [ ]:
resposta, _ = responder_com_rag("A TechNova oferece plano odontológico?")
print(resposta)

Não encontrei essa informação nos documentos.


## 6. Chatbot com RAG

Juntando tudo em um chatbot. A cada pergunta, ele **busca** no documento e **responde** com base nos trechos encontrados.

Para a conversa fluir (ex.: *"e quantos dias de antecedência?"* depois de perguntar sobre férias), guardamos as **últimas 4 mensagens** no histórico. Limitamos a 4 porque o modelo é pequeno e tem pouco espaço de contexto.

Sugestões de perguntas para a demonstração:
- *Que dias posso fazer home office?*
- *Minha senha bloqueou, o que eu faço?*
- *Quanto tenho para gastar com cursos?* → *e precisa de aprovação?*
- *Qual é o salário do CEO?* (não está no documento)

Digite `sair` para encerrar.

In [ ]:
from langchain_core.messages import AIMessage

historico = []
print("🤖 Assistente de RH da TechNova. Digite 'sair' para encerrar.\n")

while True:
    pergunta = input("Você: ").strip()
    if pergunta.lower() in {"sair", "exit", "quit"}:
        print("Até logo!")
        break
    if not pergunta:
        continue

    resposta, _ = responder_com_rag(pergunta, historico[-4:])
    print("Bot:", resposta, "\n")

    historico += [HumanMessage(content=pergunta), AIMessage(content=resposta)]

🤖 Assistente de RH da TechNova. Digite 'sair' para encerrar.

Você: Que dias posso fazer home office?
Bot: Segunda-feira e Sexta-feira. 

Você: Minha senha bloqueou, o que eu faço?
Bot: Faça uma solicitação ao HelpTech para que seu problema seja resolvido. 

Você: sair
Até logo!


## Recapitulando

| Etapa | O que fizemos | Ferramenta |
|-------|---------------|------------|
| Preparar | Dividimos o documento em pedaços | `RecursiveCharacterTextSplitter` |
| Indexar | Transformamos cada pedaço em vetor e guardamos | `HuggingFaceEmbeddings` + `InMemoryVectorStore` |
| Buscar | Encontramos os trechos mais parecidos com a pergunta | `retriever` |
| Gerar | O LLM respondeu usando só esses trechos | `ChatHuggingFace` (Qwen2.5) |

**Por que RAG é tão usado?** Para atualizar o conhecimento do chatbot, basta trocar os documentos — sem treinar o modelo de novo. E, como mostramos as fontes, as respostas podem ser verificadas.

**Para usar com seus próprios arquivos**, troque a variável `base_conhecimento` pelo conteúdo de um PDF, por exemplo com `PyPDFLoader` do pacote `langchain-community`. O resto do código continua igual.

🎉 **Parabéns!** Você construiu um chatbot com RAG do zero, 100% com modelos abertos.